# Introduction to PyTorch: From Tensors to a Neural Network for Regression

In the previous notebook we built linear regression, logistic regression, and neural networks by hand: we kept a list of weight tensors, wrote the loss formula, and subtracted gradients ourselves. That's the best way to see what's going on, but it doesn't scale to real models.

This notebook introduces the PyTorch tools that take over each of those jobs:

1. **Tensors**: creating them, shapes, broadcasting, and converting to and from NumPy
2. **Running on a GPU**: what goes on the GPU, what doesn't, and the rules to follow
3. **Linear regression**, first by hand and then with `nn.Linear`, `nn.MSELoss`, and `torch.optim`
4. **A neural network** with `nn.Sequential`, trained on mini-batches from a `DataLoader`
5. **Optimizers**: plain SGD vs momentum vs Adam
6. **Writing a model as a class** with `nn.Module`

We'll use the California housing data throughout, so you can compare with the hand-built network from the previous notebook.

The official PyTorch tutorials are a good companion: https://pytorch.org/tutorials/beginner/basics/intro.html

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

## Which device are we running on?

PyTorch can run on a CPU, an NVIDIA GPU (`"cuda"`), or an Apple Silicon GPU (`"mps"`). This cell picks the best one available, and we use the variable `device` throughout the notebook. Part 2 explains what this means and how to work with a GPU. Everything in this notebook also runs fine on a CPU.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

# Part 1: Tensors

A tensor is PyTorch's multidimensional array. If you know NumPy, most of this will look familiar.

In [ ]:
a = torch.tensor(3.0)                     # scalar (0-D)
v = torch.tensor([1.0, 2.0, 3.0])         # vector (1-D)
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])       # matrix (2-D)

print(a.shape, v.shape, M.shape)

print(torch.zeros(2, 3))
print(torch.arange(0, 10, 2))
torch.manual_seed(42)
print(torch.randn(2, 3))    # random numbers from a standard normal distribution

## Shape, dtype, and device

Every tensor has three properties you'll check constantly:

* `shape`: the size along each dimension
* `dtype`: the data type. Neural networks almost always use `torch.float32`
* `device`: where the data lives (CPU or GPU)

In deep learning, the **first dimension is almost always the batch**: the number of examples processed together. Tabular data has shape `(batch, n_features)`; grayscale images (next notebook) have shape `(batch, channels, height, width)`.

In [ ]:
print("shape :", M.shape)
print("dtype :", M.dtype)
print("device:", M.device)

ints = torch.tensor([1, 2, 3])
print("\ninteger tensor:", ints.dtype, "  converted:", ints.float().dtype)

## Indexing and reshaping

Indexing works just like NumPy. `.item()` turns a one-element tensor into a plain Python number.

`reshape` changes the shape without changing the data. Use `-1` to let PyTorch work out one dimension. A common use: turning a vector of targets with shape `(n,)` into a column with shape `(n, 1)`.

In [ ]:
X = torch.tensor([[10, 20, 30],
                  [40, 50, 60]])
print(X[0])
print(X[:, 1])
print(X[1, 2])
print(X[1, 2].item())

t = torch.arange(12)
print(t.shape, "->", t.reshape(3, 4).shape, "->", t.reshape(-1, 1).shape)

## Math operations

Elementwise operations (`+`, `*`, `exp`, ...), matrix multiplication (`@`), and reductions (`sum`, `mean`, `max`).

Reductions take a `dim` argument saying **which dimension to collapse**. `dim=0` collapses the rows, giving one value per column, which is exactly what we need to standardize features.

In [ ]:
print(M * 10)
print(M @ M.T)              # (2, 3) @ (3, 2) -> (2, 2)
print(M.mean())             # mean of everything
print(M.mean(dim=0))        # column means, shape (3,)

## Broadcasting

When two tensors have different shapes, PyTorch "stretches" dimensions of size 1 (or missing leading dimensions) to make them match. This is how `M - M.mean(dim=0)` subtracts each column's mean from that column.

In [ ]:
print(M - M.mean(dim=0))                # (2, 3) - (3,) -> (2, 3)

a = torch.tensor([[1.], [2.], [3.]])    # shape (3, 1)
b = torch.tensor([10., 20., 30., 40.])  # shape (4,)
print(a + b)                            # (3, 1) + (4,) -> (3, 4)

> ⚠️ **Broadcasting can also bite you.** Above, a `(3, 1)` tensor plus a `(4,)` tensor silently produced a `(3, 4)` result. The same thing happens if model predictions have shape `(n, 1)` and targets have shape `(n,)`: their difference is an `(n, n)` matrix. We'll see this in action shortly.

## Converting to and from NumPy

* `torch.tensor(array, dtype=torch.float32)` converts a NumPy array to a tensor. NumPy uses 64-bit floats by default, but PyTorch layers use 32-bit floats, so convert explicitly.
* `.numpy()` converts a tensor back to a NumPy array.

In [ ]:
arr = np.array([[1., 2.], [3., 4.]])
print(arr.dtype)

t = torch.tensor(arr, dtype=torch.float32)
print(t.dtype)

back = t.numpy()
print(type(back), back.dtype)

# Part 2: Running on a GPU

## Why GPUs?

A CPU has a handful of powerful cores, built to run one step after another very quickly. A GPU (graphics processing unit) has thousands of simpler cores, built to do many simple calculations at the same time. Training a neural network is mostly large matrix multiplications, where every entry of the result can be computed independently, so it's a perfect fit. For large models, a GPU can be 10 to 100 times faster than a CPU.

PyTorch names each kind of hardware with a **device** string:

| Device | Hardware |
|---|---|
| `"cpu"` | the CPU, always available |
| `"cuda"` | an NVIDIA GPU (on Linux or Windows) |
| `"mps"` | the GPU in an Apple Silicon Mac (M1 or later) |

## Getting access to a GPU

* **Your own computer**, if it has an NVIDIA GPU or is an Apple Silicon Mac. The cell at the top of the notebook detects it automatically.
* **Google Colab**: choose *Runtime → Change runtime type* and pick a GPU. Free, with time limits.
* **A shared server with several GPUs** (a JupyterHub, for example): running `nvidia-smi` in a terminal lists the GPUs and how busy they are. To use just one of them, put this at the very top of the notebook, **before** importing torch:
  ```python
  import os
  os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only GPU number 0
  ```

In [ ]:
# on the Jupyterhub
# !nvidia-smi

In [ ]:
print("device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("number of GPUs:", torch.cuda.device_count())
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif device == "mps":
    print("Apple Silicon GPU (MPS)")
else:
    print("No GPU found, so everything will run on the CPU.")

## Moving tensors to the GPU

Every tensor lives on one device. New tensors are created on the CPU unless you say otherwise. There are two ways to get a tensor onto the GPU:

* **`.to(device)`** returns a **copy** of the tensor on that device. The original doesn't change, so you have to keep the result: `x = x.to(device)`.
* **The `device=` argument** creates a new tensor directly on the device: `torch.zeros(3, device=device)`.

`.cpu()` is shorthand for `.to("cpu")`.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
print("x starts on:   ", x.device)

x.to(device)                         # makes a copy on the device... and throws it away
print("still on:      ", x.device)

x = x.to(device)                     # keep the copy
print("now on:        ", x.device)

z = torch.zeros(3, device=device)    # created directly on the device
print("z is on:       ", z.device)

print("x + z is on:   ", (x + z).device)   # results stay on the same device

## Common mistake 1: mixing devices

All the tensors in an operation must be on the same device. PyTorch never moves data for you; it raises an error instead. (Plain Python numbers, like `2` or `0.5`, are fine to mix with tensors on any device.)

In [ ]:
if device != "cpu":
    a = torch.ones(3)                    # on the CPU
    b = torch.ones(3, device=device)     # on the GPU
    try:
        a + b
    except RuntimeError as e:
        print("RuntimeError:", e)
else:
    print("We're on a CPU, so this mistake can't happen here. On a GPU, the error reads:\n"
          "  RuntimeError: Expected all tensors to be on the same device, "
          "but found at least two devices, cuda:0 and cpu!")

## Common mistake 2: handing a GPU tensor to NumPy, matplotlib, or scikit-learn

These libraries only understand data in the CPU's memory, and they can't handle tensors that are tracking gradients. Before passing a tensor to them:

* `.detach()` if it tracks gradients (or compute it inside `torch.no_grad()`)
* `.cpu()` to bring it back to the CPU
* `.numpy()` if you need a NumPy array

`.item()`, which returns a single number as a plain Python value, works from any device.

In [ ]:
w = torch.randn(3, requires_grad=True, device=device)

try:
    w.numpy()
except (RuntimeError, TypeError) as e:
    print(type(e).__name__ + ":", e)

print("w on ", w.device)
print(w.detach().cpu().numpy())    # the safe recipe, on any device
print("w on ", w.device)
print(w.sum().item())              # .item() works on any device
print("w on ", w.device)

## When is the GPU faster?

Using a GPU has overhead: copying data to it, and starting each operation. For tiny computations the CPU wins; for big ones the GPU wins by a wide margin.

Let's time matrix multiplications of different sizes. One detail: GPU operations run **asynchronously**, which means Python moves on to the next line while the GPU is still working. To time them fairly, we have to wait for the GPU to finish (`synchronize`).

In [ ]:
import time

def synchronize(dev):
    if dev == "cuda":
        torch.cuda.synchronize()
    elif dev == "mps":
        torch.mps.synchronize()

def time_matmul(n, dev, repeats=10):
    A = torch.randn(n, n, device=dev)
    B = torch.randn(n, n, device=dev)
    A @ B                                  # warm-up: the first call is always slower
    synchronize(dev)
    start = time.perf_counter()
    for _ in range(repeats):
        C = A @ B
    synchronize(dev)
    return (time.perf_counter() - start) / repeats * 1000   # milliseconds

for n in [10, 100, 1000, 2000]:
    line = f"{n:5d} x {n:<5d}  CPU: {time_matmul(n, 'cpu'):9.3f} ms"
    if device != "cpu":
        line += f"   {device}: {time_matmul(n, device):9.3f} ms"
    print(line)

On a GPU you'll typically see the CPU win for the 10×10 case and the GPU win by a lot for 2000×2000.

The housing network in this notebook is small (about 2,500 parameters and 8 input features), so a GPU won't speed it up much. The image models in the next notebook benefit more.

## What goes on the device, and what doesn't

| | Move to the device? | How |
|---|---|---|
| **The model** (its parameters) | **Yes**, once, right after creating it and **before** creating the optimizer | `model = model.to(device)` |
| **Each batch** of inputs and targets | **Yes**, inside the training loop | `X_batch, y_batch = X_batch.to(device), y_batch.to(device)` |
| **The full dataset**, `Dataset`, `DataLoader` | **No**: GPU memory is limited, so keep the data on the CPU and move one batch at a time | |
| **New tensors** created during a computation | Create them on the right device | `torch.zeros(..., device=device)` |
| **The loss function** (`nn.MSELoss()`) | **No**: it has no parameters | |
| **The optimizer** | **No**: it updates the model's parameters wherever they are. Create it *after* moving the model | |
| **Python numbers** (learning rate, number of epochs) | **No** | |
| **A single result** to print | Bring it back as a number | `loss.item()` |
| **Results** for NumPy, matplotlib, scikit-learn | Bring them back to the CPU | `preds.detach().cpu().numpy()` |

One subtlety: for a **model**, `.to(device)` moves the parameters **in place** (and also returns the model), while for a **tensor** it returns a copy. Writing `model = model.to(device)` and `x = x.to(device)` works correctly for both, so always use that form.

## Rules for working with PyTorch on a GPU

1. **Choose the device once**, at the top of the notebook, and use the `device` variable everywhere. Never type `"cuda"` directly into your code; then the same notebook runs on a CPU, an NVIDIA GPU, or a Mac.
2. **Always keep the result of `.to()`**: `x = x.to(device)`.
3. **Move the model before creating the optimizer.**
4. **Keep the data on the CPU and move each batch** to the device inside the loop.
5. **Every tensor in an operation must be on the same device.** If you see *"Expected all tensors to be on the same device"*, find the one you forgot to move.
6. **Use `float32`.** Apple GPUs don't support 64-bit floats at all, so `torch.tensor(numpy_array)` without `dtype=torch.float32` fails on a Mac.
7. **Bring results back with `.cpu()`** before using NumPy, matplotlib, or scikit-learn (and `.detach()` them, or compute them inside `torch.no_grad()`).
8. **Evaluate inside `torch.no_grad()`**: it saves GPU memory and time.
9. **If you get "CUDA out of memory"**, reduce the batch size. Restarting the kernel frees all GPU memory.
10. **Results on a GPU can differ from the CPU** in the last few decimal places. That's normal.

# Part 3: Linear regression, moving from hand-made creation to PyTorch tools

## The data

The California housing dataset: 8 features per district, and the target is the median house value in units of $100,000.

This time we split the data three ways:
* **training set**: used to fit the parameters
* **validation set**: used to monitor training and compare models and settings
* **test set**: used only once, at the very end, for an unbiased estimate of performance

In this part the data is small and everything stays on the CPU. From Part 4 on, we follow the GPU rules from Part 2.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42)

print(housing.feature_names)
print(X_train.shape, X_valid.shape, X_test.shape)

Convert to `float32` tensors and standardize, using the training set's mean and standard deviation for all three sets.

PyTorch layers produce outputs of shape `(batch, n_outputs)`, which is `(batch, 1)` for a single output. To match, we reshape the targets into columns of shape `(n, 1)`.

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)

means = X_train.mean(dim=0)
stds = X_train.std(dim=0)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test  = (X_test  - means) / stds

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).reshape(-1, 1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).reshape(-1, 1)

n_features = X_train.shape[1]
X_train.shape, y_train.shape

## By hand, as in the previous notebook

Parameters created with `requires_grad=True`, `loss.backward()` to compute gradients, an update inside `torch.no_grad()`, and resetting the gradients to zero.

In [ ]:
torch.manual_seed(42)
w = torch.randn(n_features, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.1
n_epochs = 50

for epoch in range(n_epochs):
    y_pred = X_train @ w + b                    # (n, 8) @ (8, 1) -> (n, 1)
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f"epoch {epoch:3d}   MSE = {loss.item():.4f}")

### The shape bug, demonstrated

Here's the broadcasting pitfall from Part 1. Suppose we forgot to reshape the targets, so `y_pred` has shape `(n, 1)` but the targets have shape `(n,)`:

In [ ]:
y_flat = y_train.reshape(-1)          # shape (n,), the "forgot to reshape" version
with torch.no_grad():
    y_pred = X_train @ w + b          # shape (n, 1)
    print("y_pred:", tuple(y_pred.shape), "  y_flat:", tuple(y_flat.shape))
    print("difference:", tuple((y_pred - y_flat).shape), "  <- an n x n matrix!")
    print("wrong MSE:  ", ((y_pred - y_flat) ** 2).mean().item())
    print("correct MSE:", ((y_pred - y_train) ** 2).mean().item())

No error, just a meaningless number. When a model trains strangely, **check your shapes first**.

## With PyTorch's tools

Three tools replace our hand-written code:

* **`nn.Linear(in_features, out_features)`** is a layer that holds a weight matrix and a bias vector, initializes them sensibly, and computes `x @ W.T + b`
* **`nn.MSELoss()`** computes the mean squared error
* **an optimizer**, `torch.optim.SGD(model.parameters(), lr=...)`, knows which tensors to update. `optimizer.step()` does the update and `optimizer.zero_grad()` resets the gradients

In [ ]:
n_features

In [ ]:
torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

for name, param in model.named_parameters():
    print(name, tuple(param.shape))

model(X_train[:3])      # calling the layer runs its computation

The training loop becomes five lines that look the same for every model you'll ever train:

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(n_epochs):
    y_pred = model(X_train)              # 1. forward pass
    loss = loss_fn(y_pred, y_train)      # 2. compute the loss
    optimizer.zero_grad()                # 3. reset old gradients
    loss.backward()                      # 4. backward pass
    optimizer.step()                     # 5. update the parameters
    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f"epoch {epoch:3d}   MSE = {loss.item():.4f}")

| By hand | PyTorch |
|---|---|
| `w = torch.randn(..., requires_grad=True)`, `b = ...` | `model = nn.Linear(n_in, n_out)` |
| `y_pred = X @ w + b` | `y_pred = model(X)` |
| `((y_pred - y) ** 2).mean()` | `nn.MSELoss()(y_pred, y)` |
| `with torch.no_grad(): w -= lr * w.grad; ...` | `optimizer.step()` |
| `w.grad.zero_(); b.grad.zero_()` | `optimizer.zero_grad()` |

To make predictions, use `torch.no_grad()` since we don't need gradients. Here's how the linear model does on the validation set:

In [ ]:
with torch.no_grad():
    valid_mse = loss_fn(model(X_valid), y_valid)
print(f"Linear model   validation MSE: {valid_mse:.4f}")

# Part 4: A neural network with `nn.Sequential` and mini-batches

## The model

`nn.Sequential` chains layers together: the output of each becomes the input of the next. Here is a network with two hidden layers of 50 and 40 units and ReLU activations, the same structure we built by hand in the previous notebook.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1),
)
model

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params, " =", (8*50 + 50), "+", (50*40 + 40), "+", (40*1 + 1))

## Mini-batches with `Dataset` and `DataLoader`

In the previous notebook we wrote a `minibatches` generator that shuffled indices and sliced out batches. PyTorch provides this as two pieces:

* a **`Dataset`** knows how many examples there are and how to fetch example number `i`. `TensorDataset(X, y)` wraps tensors we already have.
* a **`DataLoader`** serves up batches from a dataset, shuffling at the start of each epoch if we ask it to.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=256)

X_batch, y_batch = next(iter(train_loader))
print("batches per epoch:", len(train_loader))
print("one batch:", X_batch.shape, y_batch.shape)

## Training and evaluation functions

We'll write these once and reuse them for the rest of this notebook, and (with small additions) in the next one.

* `train_one_epoch` runs the five-line loop over every batch, and returns the average loss.
* `evaluate` computes the average loss on a dataset, inside `torch.no_grad()`.
* `fit` calls both once per epoch and records the results.
* **`model.train()` and `model.eval()`** switch the model between training and evaluation mode. Our models behave the same either way, but some layer types behave differently, so it's a good habit to always set the mode.
* Following the GPU rules from Part 2: each batch is moved to `device` inside the loop, while the data itself stays on the CPU. The model is moved to `device` when it's created (in `make_mlp` below), before the optimizer is created.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        loss = loss_fn(model(X_batch), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total_loss += loss_fn(model(X_batch), y_batch).item() * len(X_batch)
    return total_loss / len(loader.dataset)


def fit(model, train_loader, valid_loader, loss_fn, optimizer, n_epochs, verbose=True):
    history = {"train_loss": [], "valid_loss": []}
    for epoch in range(n_epochs):
        history["train_loss"].append(train_one_epoch(model, train_loader, loss_fn, optimizer))
        history["valid_loss"].append(evaluate(model, valid_loader, loss_fn))
        if verbose:
            print(f"epoch {epoch + 1:3d}/{n_epochs}   train loss {history['train_loss'][-1]:.4f}"
                  f"   valid loss {history['valid_loss'][-1]:.4f}")
    return history

(Each batch's loss is multiplied by its size before adding up, because the last batch may be smaller than the others.)

Now train the network with mini-batch SGD:

In [ ]:
def make_mlp(seed=42):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(n_features, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU(),
        nn.Linear(40, 1),
    ).to(device)                       # move the model to the device...

model = make_mlp()
# ...and only then create the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
history_sgd = fit(model, train_loader, valid_loader, nn.MSELoss(), optimizer, n_epochs=20)

## Learning curves

Plotting the training and validation loss against epochs is the most useful diagnostic for a neural network:

* both still falling → train longer
* training loss falls while validation loss rises → **overfitting**
* both flat and high → the model is too simple, or the learning rate is off

In [ ]:
epochs = range(1, 21)
plt.figure(figsize=(6, 4))
plt.plot(epochs, history_sgd["train_loss"], ".--", label="training")
plt.plot(epochs, history_sgd["valid_loss"], ".-", label="validation")
plt.xticks(range(0, 21, 5))
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Learning curves: neural network, mini-batch SGD")
plt.show()

The neural network's validation MSE is already well below the linear model's.

# Part 5: Better optimizers

Plain SGD takes a step of size `lr × gradient` every time. Two widely used improvements:

* **Momentum** keeps a running average of past gradients and moves in that direction, like a ball rolling downhill that builds up speed. This smooths out noisy mini-batch gradients. Because it keeps adding up past gradients, its effective step is larger, so it's usually paired with a smaller learning rate. `torch.optim.SGD(..., momentum=0.9)`
* **Adam** adds momentum, plus a separate, automatically adapted step size for each parameter, based on how large its recent gradients have been. It works well on many problems with little tuning, which makes it the most common default. `torch.optim.Adam(..., lr=0.001)`

Swapping optimizers is a one-line change. Everything else stays the same.

In [ ]:
histories = {"SGD": history_sgd}

model = make_mlp()
histories["SGD + momentum"] = fit(model, train_loader, valid_loader, nn.MSELoss(),
                                  torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9),
                                  n_epochs=20, verbose=False)

model = make_mlp()
histories["Adam"] = fit(model, train_loader, valid_loader, nn.MSELoss(),
                        torch.optim.Adam(model.parameters(), lr=0.001),
                        n_epochs=20, verbose=False)

plt.figure(figsize=(6, 4))
for name, h in histories.items():
    print(f"{name:15s} best validation MSE: {min(h['valid_loss']):.4f}")
    plt.plot(epochs, h["valid_loss"], ".-", label=name)
plt.xticks(range(0, 21, 5))
plt.ylim(0.25, 0.6)
plt.xlabel("epoch"); plt.ylabel("validation MSE"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Validation loss by optimizer")
plt.show()

# Part 6: Writing a model as a class

`nn.Sequential` is convenient, but most PyTorch code you'll read defines models as **classes** that inherit from `nn.Module`. This gives you full control over how data flows through the model.

A quick refresher on Python classes:

* **`__init__`** runs when the model is created. That's where we create the layers.
* **`self`** refers to the model itself, so `self.hidden1` is a layer belonging to this model.
* **`super().__init__()`** runs `nn.Module`'s own setup code. Always call it first.
* **`forward(self, x)`** describes the computation. Calling `model(x)` runs `forward`, so you never call `forward` directly.

Here's the same network as before, written as a class:

In [ ]:
class RegressionMLP(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.hidden1 = nn.Linear(n_inputs, 50)
        self.hidden2 = nn.Linear(50, 40)
        self.output = nn.Linear(40, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

torch.manual_seed(42)
model = RegressionMLP(n_features).to(device)
model

PyTorch finds every layer assigned in `__init__`, so `model.parameters()` works automatically, and training works exactly as before. Let's train it with Adam for 30 epochs:

In [ ]:
history = fit(model, train_loader, valid_loader, nn.MSELoss(),
              torch.optim.Adam(model.parameters(), lr=0.001), n_epochs=30, verbose=False)

print(f"best validation MSE: {min(history['valid_loss']):.4f}")

## Final evaluation on the test set

We've used the validation set to make choices. Now, once, we check the final model on the test set. We report the root mean squared error (RMSE), which is in the same units as the target, and R².

In [ ]:
from sklearn.metrics import r2_score

model.eval()
with torch.no_grad():
    test_pred = model(X_test.to(device)).cpu()   # inputs to the device, predictions back to the CPU

test_rmse = ((test_pred - y_test) ** 2).mean().sqrt().item()
print(f"Test RMSE: {test_rmse:.3f}  (about ${test_rmse * 100_000:,.0f})")
print(f"Test R^2:  {r2_score(y_test.numpy(), test_pred.numpy()):.3f}")

plt.figure(figsize=(5, 5))
plt.scatter(y_test, test_pred, s=3, alpha=0.3)
plt.plot([0, 5], [0, 5], "r--")
plt.xlabel("actual ($100k)"); plt.ylabel("predicted ($100k)")
plt.title("Neural network: test set predictions")
plt.show()

# Summary

The complete PyTorch workflow for a supervised learning problem:

```python
# data
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

# device: chosen once, at the top
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# model (moved to the device first), loss, optimizer
model = nn.Sequential(nn.Linear(n_in, 50), nn.ReLU(), nn.Linear(50, 1)).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# training loop
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        loss = loss_fn(model(X_batch), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# predictions, brought back to the CPU
model.eval()
with torch.no_grad():
    predictions = model(X_test.to(device)).cpu()
```

In the next notebook we apply exactly this workflow to images: classifying handwritten digits, first with a dense network and then with a convolutional neural network.